---
title: Natural Language Processing
jupyter: python3
execute:
  cache: true
---

## Introduction

In our world, Natural Language Processing (NLP) is used in several scenarios. For example, 

* mobile phones and personal computers support predictive text.
* web search engines give access to information locked up in unstructured text; 
* machine translation allows us to understand texts written in languages that we do not know; 
* text analysis enables us to detect sentiment in tweets and blogs.

But as we begin to explore NLP, we realise that it is an extremely difficult
subject. Here are some examples to note:

1.  Some words mean different things in different contexts, but as humans, we
    know which meaning is being used.
    * He **served** the **dish**.
3.  In the following two sentences, the word "by" has different meanings:
    * The lost children were found by the lake.
    * The lost children were found by the search party.
3.  In the following cases, we (humans) can resolve what "they" is referring to,
    but it is not easy to generate a simple rule that a computer can follow.
    * The thieves stole the paintings. They were subsequently recovered.
    * The thieves stole the paintings. They were subsequently arrested.
4.  How can we get a computer to understand that the following tweet carries a 
    negative sentiment?
    * "Wow. Great job st@rbuck's. Best cup of coffee ever."
    
::: {.callout-note}

Can you catch all three jokes in the movie clip below? 🤣

{{< video https://youtu.be/NfN_gcjGoJo  aspect-ratio="4x3" align="center">}}

:::


In [1]:
import numpy as np
import pandas as pd

from itables import show
import pprint
import os

#import gensim
#from gensim.parsing.preprocessing import *
#import gensim.downloader as api
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

#nltk.download('stopwords')
#nltk.download('punkt')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn import manifold
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import LatentDirichletAllocation

from transformers import pipeline
from staticvectors import StaticVectors

#import pyLDAvis
#import pyLDAvis.gensim_models as gensimvis

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

## Definitions

Before we go on, it would be useful to establish some terminology:


* A **corpus** is a collection of documents. 
  * Examples are a group of movie reviews, a group of essays, a group of paragraphs,
    or just a group of tweets.
  * Plural of corpus is **corpora**.
* A **document** is a single unit within a corpus.
  * Depending on the context, examples are a single sentence, a single paragraph,
    or a single essay.
* **Terms** are the elements that make up the document. They could be individual 
  words, bigrams or trigrams from the sentences. These are also sometimes referred to as **tokens**.
* The **vocabulary** is the set of all terms in the corpus.

Consider the sentence:

```
I am watching television.
```

The process of splitting up the document into tokens is known as tokenisation. The result for the above sentence would be

```
'I',  'am',  'watching', 'television', '.'
```

How we tokenize and pre-process things will affect our final results. We shall discuss this more in a minute.

## Overview of Applications

Here are some of the use-cases that we shall discuss:

1. *Topic Modeling*: This is an unsupervised technique that allows us to
    identify the salient topics of a new document automatically. This could be
    useful in a customer feedback setting, because it would allow quick allocation
    or prioritisation of resources. This approach requires one to decide on the
    number of topics. It typically also requires some study of the topics in order
    to interpret and verify them.
2. *Information Retrieval*: This is also an unsupervised approach. Suppose we have 
    a collection of documents. A new document, considered a "query", can be used to 
    retrieve documents from the collection that are relevant to the query.
3. *Sentiment Analysis*: This technique is used to assess whether the sentiment
    in a document is mostly positive or negative.

## Text Pre-processing

::: {#exm-wine-reviews-1 style="background-color: #D5D1D164; padding: 20px"}

### Wine reviews dataset
\index{Wine Reviews!Description}

A dataset containing wine reviews is accessible from
[Kaggle](https://www.kaggle.com/zynicide/wine-reviews). We shall work with one
of the csv files. It contains 130,000 rows, although some are duplicates.

In [2]:
rng = np.random.default_rng(5001)

wine_reviews = pd.read_csv("data/winemag-data-130k-v2.csv", index_col=0)
wine_reviews.drop_duplicates(inplace=True)

The `description` column contains the review for a particular wine by a user,
whose name and twitter handle are provided. Also included is information such
as the price, originating county, region of the wine, and so on. In this chapter,
we are going to apply NLP techniques to the wine reviews.

Here is a sample of some of reviews in the dataset.

In [4]:
pp = pprint.PrettyPrinter(indent=4, compact=True,)
for x in rng.choice(wine_reviews.description, size=5):
    pp.pprint(x)

('Bell pepper and sharp red fruit aromas provide a shaky start, which is '
 'followed by cranberry, tart cherry and other pointed flavors. The feel is '
 'racy and tight, with gritty acids. Airing does improve it somewhat. Tasted '
 'twice; this is a review of the better bottle. Cabernet, Merlot, Cab Franc '
 'and Carmenère is the blend. From Brazil.')
('Consistent with previous releases, this Michel Rolland effort is a soft, '
 'silky, smoky wine that introduces itself with round cherry fruit and then '
 'charges ahead with layers of licorice, citrus, coffee and rock that enliven '
 'the finish. There is plenty of tart raspberry fruit to open, and the '
 "balancing acids to give the wine a tight core. It's a very polished and "
 'appealing balance of forward, approachable fruit married to more elegant, '
 'ageworthy tannins and acids.')
("This is a light and soft selection, with an upfront gamy note that's framed "
 'by soft strawberry, rhubarb, red cherry and currant fruit tones on t

:::

### Pre-processing Text with Gensim

Text documents consist of sentences of varying lengths. Usually, the first step
to analysing a document is to break it up into pieces. This process is known as
**tokenizing**. When tokenizing a document, we can do it at several levels of
resolution: at the sentence, line, word or even punctuation level.

Tokenizing can easily done using the `.split()` method, which is built into
python. But after that, we need to further pre-process the tokens.

The `gensim` package includes a module for pre-processing text strings. Here is
a list of some of the functions there:

* `strip_multiple_whitespaces`
* `strip_non_alphanum`
* `strip_numeric`
* `strip_punctuation`
* `strip_short`

Since what we are about to do in the initial part of our activity is based on
frequency counts of tokens, apart from some of the above steps, we are also
going to remove common "filler" words that could end up skewing the eventual
probability distributions of counts. These filler words are known as stop
words. They were identified by linguists, and they vary from model to model,
from Python package to package, and of course, from language to language.

Whether stop-word removal is meaningful or not also depends on your particular
application. At times, it is only done in order to speed up the training of a
model. However, it is possible to change the entire meaning of a sentence by
removing stop-words.

For us, we are going to apply this list of filters to each wine review:

1. `strip_punctuation()`,
2. `strip_multiple_whitespaces()`,
3. `strip_numeric()`,
4. `remove_stopwords()`,
5. `strip_short()`,
6. `lemmatize()`

**Lemmatizing** a word is to reduce it to its root word. You will come across
**stemming** whenever you read about lemmatizing. In both cases, we wish to
reduce a word to its root word so that we do not have to deal with multiple
variations of a token, such as ate, eating, and eats.

When we stem a word, the prefix and/or suffix will be removed according to a
set of rules. Since it is primarily rule-based, the resulting word may not be
an actual English word.

Like stemming, lemmatizing also aims to reduce a word to its root form.
However, it differs from stemming in that the final word must be a proper
English language word. For this purpose, the algorithm has to be supplied with
a lexicon or dictionary, along with the text to be lemmatized.

::: {#exm-stem-lemmatise style="background-color: #D5D1D164; padding: 20px"}

### Stemming versus lemmatizing

Here is an example that demonstrates the differences between stemming and 
lemmatizing. Consider the following simple sentence.

In [ ]:
demo_sentence = 'Cats and ponies have a meeting'.split()
demo_sentence

This is the outcome of stemming the words in this sentence:

In [ ]:
porter = PorterStemmer()
[porter.stem(x) for x in demo_sentence]

If you have not previously done so, you will need to download the Wordnet
lemmatizer.

In [ ]:
#| eval: false
import nltk
#nltk.download('wordnet')
#nltk.download('stopwords')
#nltk.download('punkt')

In [ ]:
en_stop_words = stopwords.words('english')

Here is the output of lemmatizing the words instead:

In [ ]:
wn = WordNetLemmatizer()
[wn.lemmatize(x) for x in demo_sentence]

:::

Now let us go ahead and perform the pre-processing on the wine reviews.

In [ ]:
x = all_review_strings[0]
x

In [ ]:
x.lower()

In [ ]:
import string

string.punctuation

In [ ]:
#text.translate(str.maketrans('', '', string.punctuation))

In [ ]:
def custom_preprocessor(x, min_length = 3):
    output = x.lower()
    output = output.translate(str.maketrans('', '', string.punctuation + '0123456789'))
    tokens = word_tokenize(output)
    output_list = [y for y in tokens if y not in en_stop_words and len(y) > min_length]
    return output_list

In [ ]:
custom_preprocessor(x)

In [ ]:
CUSTOM_FILTER = [lambda x: x.lower(), strip_punctuation, 
                 strip_multiple_whitespaces, strip_numeric, 
                 remove_stopwords, strip_short]
#CUSTOM_FILTER[1]

The object `CUSTOM_FILTER` is a collection of functions that will be applied 
to each wine review.

In [6]:
all_review_strings = wine_reviews.description.values

In [ ]:
all_strings_tokenized = [custom_preprocessor(x) for x in all_review_strings]

At this point in time, what we have is a list of lists. Each sub-list contains
the tokens for a particular wine_review. For instance, the original review in 
row 234 is:

In [ ]:
pp.pprint(wine_reviews.description.values[233])

The corresponding processed output is:

In [ ]:
pp.pprint(all_strings_tokenized[233])

`gensim` does not have a lemmatizer, so we use the Wordnet lemmatizer on each
token.

In [ ]:
preprocessed_corpus = [[wn.lemmatize(w) for w in dd ] for dd in all_strings_tokenized]

## Representation of Text {#sec-04-numeric-rep-text}

Tokenisation of the corpus is merely the first step in processing natural
language. All mathematical algorithms work on numerical representations of the
data, so the next step is to convert the text into numeric representations. In
natural language, there are two common ways of representing text:

1.  Sparse vectors, using tf-idf or PPMI, or
2.  Dense embeddings, which could result from word2vec, GLoVe, or from neural
    language models.

### Sparse embeddings with Tf-idf

In this section, we demonstrate how we can use Tf-idf (Term frequency-Inverse
document frequency) to create vector representations of documents. 

::: {#exm-tfidf-01 style="background-color: #D5D1D164; padding: 20px"}

### Term frequencies

Consider the following set of three simple text documents. Each document is a
single sentence.

In [ ]:
raw_docs =[
  "Here are some very simple basic sentences.", 
  "They won’t be very interesting , I’m afraid. ",  
  """
  The point of these basic examples is to learn how basic text  
  counting works on *very simple* data, so that we are not afraid when  
  it comes to larger text documents. The sentences are here just to provide words.
  """] 

As the name tf-idf suggests, our first step should be to compute the frequency
of each term (token) within each document.

In [ ]:
vectorizer1 = CountVectorizer(stop_words='english', min_df=1)
X1 = vectorizer1.fit_transform(raw_docs)

print(pd.DataFrame(X1.toarray(), 
  columns=vectorizer1.get_feature_names_out()).iloc[:, :10])

The counts indicate the number of times each feature (or words) were present in
the document. 

:::

As you might observe, longer documents tend to contain larger counts (see
document 3, which has many more 1's and even a couple of 2's. Thus, instead of
dealing with counts, we shall convert each row into a vector of length 1. Words
that appear in all documents will be weighted down by this transformation, since
these do not help to distinguish the document from others. This transformation
is known as the TF-IDF transformation.

Instead of the raw counts, we define:

* $N$ to be the number of documents ($N=3$ in the little example above).
* $tf_{i,j}$ to be the frequency of term $i$ in document $j$.
* $df_{i}$ to be the frequency of term $i$ across all documents.
* $w'_{i,j}$ to be: 

\begin{equation}
w'_{i,j} = tf_{i,j} \times \left[ \log \left( \frac{1 + N}{1 + df_{i}} \right) + 1 \right]
\end{equation}

Then the final $w_{i,j}$ for term $i$ in document $j$ is the normalised version
of $w'_{i,j}$ across the terms that document. 

::: {#exm-tfidf-02 style="background-color: #D5D1D164; padding: 20px"}

### Tf-idf computation

Consider the word "sentences", in document id 02 (the third document).

* $N = 3$
* $tf_{i,j} = 1$
* $df_{i} = 2$

Thus

\begin{equation}
w'_{ij} = 1 \times \log ( (1+3)/(1 +2)) = 1.287
\end{equation}

In [ ]:
wine_reviews.description[0]

In [ ]:
vectorizer2 = TfidfVectorizer(stop_words='english', norm=None)
X2 = vectorizer2.fit_transform(raw_docs)

In [ ]:
X2.toarray()

In [ ]:
print(pd.DataFrame(X2.toarray(), 
  columns=list(vectorizer2.get_feature_names_out())).iloc[:, :10].round(3))

The final step normalises the weights within each document. 

In [ ]:
vectorizer3 = TfidfVectorizer(stop_words='english')
X3 = vectorizer3.fit_transform(raw_docs)

print(pd.DataFrame(X3.toarray(), 
  columns=list(vectorizer3.get_feature_names_out())).iloc[:, :10].round(3))

:::

The above matrix is known as a **document-term matrix**, since the columns are
defined by terms, and each row is a document. At this point, we can use each
**row** as a vector representation of each document. If necessary, for this
corpus, we could even represent each term using its corresponding **column**.

Note that some books/software use a slightly different convention - they may
work with the *term-document* matrix. However, the idea is the same. Take a
look at the following term-document matrix, assembled from the complete works
of Shakespeare:

![@jm3](figs/term-doc-shakespeare-1.png){width=70%}

If we intend to represent each document as a numeric vector, the columns,
highlighted by the red boxes, would be a natural choice. Suppose we only focus
on the coordinates corresponding to the words `battle` and `fool`. Then a
visualisation of the documents would look like this:

![@jm3](figs/term-doc-shakespeare-2.png){width=70%}

Visually, it is easy to tell that "Henry V" and "Julius Caesar" are similar
(they point in the same direction) as opposed to "As You Like It" and "Twelfth
Night". But it is also easy to see *why* - the former two contain similar high
counts of `battle` compared to the latter two, which are comedies. 

Tf-idf are a normalised version of the above raw counts; they provide a
numerical representation of documents, adjusting for document length and words
that are common across all documents in a corpus.

### Cosine similarity

In order to quantify the similarity (or nearness) of vector representations in
NLP, the common method used is cosine similarity. Suppose that we have a vector
representation of two documents $\mathbf{v}$ and $\mathbf{w}$. If the
vocabulary size is $N$, then each of the vectors is of length $N$. Since we are
dealing with counts the coordinate values of each vector will be non-negative.
We use the angle $\theta$ between the vectors as a measure of their similarity:

$$
\cos \theta = \frac{\sum_{i=1}^N v_i w_i}{\sqrt{\sum_{i=1}^N v_i^2} \sqrt{\sum_{i=1}^N w_i^2}}
$$

Geometrically, cosine similarity measures the size of the angle between vectors:

![@jm3](figs/term-doc-shakespeare-4.png){width=70%}

### Dense Embeddings

One of the drawbacks of sparse vectors is that they are very long (the length
of the vocabulary), and most entries in the vector will be 0. As a result,
researchers worked on methods that would pack the information in the vectors
into shorter ones. Instead of working on representations of the documents, the
methods aimed to create representations of each token (or word) in the
vocabulary. These are referred to as *embeddings*.

Here, we shall discuss word2vec (@mikolov2013distributed), but take note that
there are others. GLoVe (@pennington2014glove) was invented soon after, but the
most common embeddings used today arise from Deep Learning models. The most
widely used version is BERT (see the video references below, as well as 
@devlin2019bert).

The approach in word2vec deviates considerably from tf-idf, in that the goal is
to obtain a numeric representation of a word, 
*in the context of it's surrounding words*. Consider this statement:

> 13% of the United States population eats <font color="red">pizza</font> on any given day. Mozzarella is commonly used on
> <font color="red">pizza</font>, with the highest quality mozzarella from Naples. In Italy, <font color="red">pizza</font> served in formal > settings is eaten with a fork and knife.

The words `eats`, `served` and `mozzarella` appear close to `pizza`. Hence
another word that appears in similar contexts, should be *similar* to `pizza`.
Examples could be certain baked dishes or even `salad`. 

To achieve such a representation, word2vec runs a self-supervised algorithm,
with two tasks:

1. **Primary task:** To "learn" a numeric vector that represents each word.
2. **Pretext task (stepping stone):** To train a classifier that, when given a word $w$, predicts nearby context words $c$.

Self-supervised algorithms differ from supervised algorithms in that there are
no labels that need to be created. The pre-text task trains a model to perform
predictions, based on a sliding window context:

::: {layout-ncol=1}
![](figs/word2vec-001a.png){width=70%}

![](figs/word2vec-001b.png){width=70%}
:::


Starting with an initial random vector for each word, the algorithm updates the
vectors as it proceeds through the corpus, finally ending up with an embedding
for each word that reflects its semantic value, based on neighbouring words.

In NLP, the quality of an embedding can be evaluated using an analogy task:

> Given X, Y and Z, find W such that W is related to to Z in the same way that X is related to Y.

For instance, if we are given the pair `man`:`king`, and the word `woman`, then
the embedding should return `queen`, since `woman`:`queen` in the same way that
`man` is related to `king`. Geometrically, the answer to the analogy is
obtained by adding (king - man) to woman. The nearest embedding to the result,
is returned as the answer.

On the left are examples of the types of analogy pairs that word2vec is able to
solve, while on the right, we have visualisations of GLoVe.


::: {layout-ncol=2}

![word2vec](figs/word2vec-relationships.png){width=70%}

![GloVE](figs/glove-001.jpg){width=70%}
:::

::: {#exm-glove-01 style="background-color: #D5D1D164; padding: 20px"}


### Glove dense embeddings

Here's how we can use `gensim` code to conduct the analogy task. First, we load
400,000 GLoVe vectors, each representing a different word. Each vector is of
length 100.

In [ ]:
import os
from sklearn.neighbors import NearestNeighbors

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
# Load GloVe from Hugging Face Hub
word_vectors = StaticVectors("neuml/glove-6B")

#all_words = list(model.word_index.keys())
#all_embeddings = word_vectors.vectors
    
# Normalize all embeddings for cosine distance
#all_embeddings_normalized = all_embeddings / np.linalg.norm(all_embeddings, axis=1, keepdims=True)

nbrs = NearestNeighbors(n_neighbors=5, metric='cosine', algorithm='auto').fit(word_vectors.vectors)

#distances, indices = nbrs.kneighbors(word_vectors.embeddings(['hello']).reshape(1, -1))
#distances, indices
#list(word_vectors.tokens.items())[13075][0]

In [ ]:
def word_analogy(model, nn_obj, word_a, word_b, word_c):
    """
    Solve the analogy: word_a is to word_b as word_c is to ____
    
    Using sklearn's NearestNeighbors (KNN) to find the closest embedding.
    
    Example: word_a="man", word_b="woman", word_c="king"
    This solves: "man is to woman as king is to ____"
    
    Algorithm:
    1. Get embeddings for word_a, word_b, word_c
    2. Compute relationship: word_b - word_a
    3. Apply to word_c: word_c + relationship = target_vector
    4. Use KNN to find nearest neighbors to target_vector
    5. Return top matches (excluding query words)
    
    Parameters:
    -----------
    model : StaticVectors
        The GloVe embedding model
    word_a : str
        First word in the analogy (e.g., "man")
    word_b : str
        Second word in the analogy (e.g., "woman")
    word_c : str
        Third word in the analogy (e.g., "king")
    topn : int
        Number of closest matches to return
    
    Returns:
    --------
    results : list of tuples
        List of (word, distance_score) pairs, sorted by proximity
    """
    
    # Step 1: Get embeddings for the three words
    try:
        emb_a = model.embeddings([word_a])[0]  # man
        emb_b = model.embeddings([word_b])[0]  # woman
        emb_c = model.embeddings([word_c])[0]  # king
    except Exception as e:
        print(f"Error: One of the words not found in vocabulary: {e}")
        return []

    #org_indices
    
    # Step 2: Compute the relationship vector (word_b - word_a)
    # This captures what it means to go from "man" to "woman"
    relationship = emb_b - emb_a
    
    # Step 3: Add the relationship to word_c
    # king + (woman - man) = target vector
    target_vector = emb_c + relationship
    
    # Normalize the target vector
    # target_vector = target_vector / np.linalg.norm(target_vector)
    
    # Step 5: Query KNN to find nearest neighbors
    distances, indices = nn_obj.kneighbors(target_vector.reshape(1, -1), n_neighbors=5)

    all_tokens = list(model.tokens.items())
    if indices is not None:
        out_tokens = [all_tokens[ii][0] for ii in indices[0]]
        #out_tokens.append(all_tokens[indices[0]
    #    return list(model.tokens.items())[indices.item()][0]
    to_return  = [xxx for xxx in out_tokens if xxx not in [word_a, word_b, word_c]]
    
    return to_return


In [ ]:
word_analogy(word_vectors, nbrs, 'switzerland', 'swiss', 'cambodia')

In [ ]:
word_vectors.vectors.shape

In [ ]:
#word_vectors = api.load("glove-wiki-gigaword-100")

The `word_vectors` object is similar to a dictionary. For instance,
`word_vectors['woman']` will return the vector corresponding to "woman".

The following code will run the analogy task. It returns the answer to:

> Man is to king as woman is to ______ .

In [ ]:
# Check the "most similar words", using the default "cosine similarity" measure.
result = word_vectors.most_similar(positive=['woman', 'king'], negative=['man'])
most_similar_key, similarity = result[0]  # look at the first match
print(f"{most_similar_key}: {similarity:.4f}")

The object also contains methods to identify which word in a group is most 
dissimilar to the rest. The following code identifies "cereal" as the odd word out.

In [ ]:
print(word_vectors.doesnt_match("breakfast cereal dinner lunch".split()))

:::

## Visualisation with t-SNE {#sec-04-t-SNE}

When compared with sparse embeddings, dense embeddings are compact. However, a
more important difference is that dense vectors contain the semantic meaning
of words. This means that vectors that are close to each other are similar
in meaning. Let us use t-SNE to visualise the GloVe embeddings.

There are a total of 400,000 vectors in the embedding. Even with t-SNE that will
be difficult to make sense of. Hence for now, we simply visualise the first 100
most common words.

The following code extracts the first 100 word vectors and applies the t-SNE 
transformation to them. Please refer to @sec-03-t-SNE for more details.

In [ ]:
nn = 1000

In [ ]:
X = np.zeros((nn, 100))
for ii in np.arange(nn):
    X[ii,] = word_vectors.get_vector(ii)

In [ ]:
X = word_vectors.vectors[:1000, :]

In [ ]:
X.shape

In [ ]:
list(word_vectors.tokens.keys())[:1000]

In [ ]:
labels = pd.Series(list(word_vectors.tokens.keys())[:nn])

In [ ]:
tsne1 = manifold.TSNE(n_components=2, init="random", perplexity=10, 
                      metric='cosine', verbose=0, max_iter=5000, 
                      random_state=222)
X_transformed2 = tsne1.fit_transform(X)

You should get the same plot as us since we have set the same seed at the start
of the cell, and when we initialise the transformer. Explore the resulting plot
- notice how months of the year appear close together at the bottom left.
Around the left as well, the calendar years appear as a group.

::: {.content-visible when-format="html"}

In [ ]:
#| fig-align: center
df2 = pd.DataFrame(X_transformed2, columns=['x','y'])
df2['labels'] = labels
fig = px.scatter(df2, x='x', y='y', text='labels', width=1024, height=960)
fig.update_traces(textposition='top center')
fig.show()

:::

::: {.content-visible when-format="pdf"}

Please refer to the online version of the text for an interactive plot, with 
text labels.

In [ ]:
#| fig-align: center
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df2, x="x", y="y")

:::

::: {.callout-note}

Would we be able to make such a plot using tf-idf? Why or why not?

:::

## Neural Language Models

Language is complex. It is incredible how we can understand such long paragraphs
of texts with such ease. We somehow seem to have learnt complicated sets of
grammar and syntax just by listening to others speak. To get a machine to learn
language has not been easy. It is only recently that Large Language Models such
as chatGPT have demonstrated that it is possible for machines to converse with
humans just as we do to one another.

Neural Models (or deep learning models) have been the key to this. In this
subsection, we provide a very brief overview of their characteristics that
allow them to achieve impressive performance on a range of language-related
tasks.

The basic unit of a neural model is the neural unit (on the left). It consists
of weights and a non-linear activation function. Given an input vector, the
weights are multiplied by the input vector, summed and then fed through the
activation function to generate an output.

::: {layout-ncol=2}
![@jm3](figs/j_m_fig_7_2_neural_unit.png){width=70%}

![Neuron figure from <https://en.wikipedia.org/wiki/Neuron>](figs/Blausen_0657_MultipolarNeuron.png){width=70%}
:::


Neural models are made up of many neural units, organised into layers. The
first neural models were Feed-Forward Networks. Due to the virtue of being able
to incorporate many parameters, and due to semi-supervised learning, they were
already a huge improvement over earlier models. Here is a simple set up, with
one hidden layer for training a language model (used to predict the next word).
It can also be used to learn embeddings.

![@jm3, FFN](figs/j_m_fig_7_17_ffn_training.png)

The next evolution in neural models was the ability to incorporate words in the
recent history. For humans, this comes naturally. For instance, we know that
this is grammatically correct:

> The flights the airline was cancelling **were** full.

For neural models to have this ability, it was necessary to incorporate the
hidden layers from recent words when processing the current word. Recurrent
Neural Networks (RNNs) and Long-Short Term Memory (LSTM) networks had these
features, but they were very slow to train. The major breakthrough came with
the invention of the transformer architecture. The self-attention layer of
these networks gave a word access to *all* preceding words in the training
window, instead of just one. Most importantly. the training of these networks
could be parallelised!

![@jm3](figs/j_m_fig_10_18_training_transformer_lm.png){width=70%}

Here are some more examples where transformers excel:

> The keys to the cabinet *are* on the table.
> 
> The chicken crossed the road because *it* wanted to get to the other side.
> 
> I walked along the pond, and noticed that one of the trees along the *bank*
> had fallen into the water after the storm.

In the final sentence, the word `bank` has two meanings - how will a model know
to decide the correct one? With transformers, because the full context of a
word is captured along with it, it is possible to perform this disambiguation.


![@jm3](figs/j_m_fig_11_8_wsd.png){width=80%}

## Applications

[Hugging Face](https://huggingface.co/) has spent a considerable effort to make
Neural Language Models accessible and available to all with minimal coding. For
starters, they have ensured that all their models are described in a
standardised manner with model cards. Here is an example of a 
[model card for BERT](https://huggingface.co/google-bert/bert-base-uncased).

Moreover, they have developed easy to use pipelines. For NLP, the following
tasks have mature pipelines:

* feature-extraction (obtaining the embedding of a text)
* ner
* question-answering
* sentiment-analysis
* summarization
* text-generation
* translation, and
* zero-shot-classification.
  
### Sentiment Analysis

In this subsection, we shall utilise one of their sentiment analysis models on
the wine reviews dataset. This is a transformer-based neural language model
(BERT) that has been fine-tuned with data labelled with sentiments. All we have
to do is feed in the sentence, and we will obtain a confidence score, and a
sentiment label.

In [ ]:
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
  
classifier(["I love this course!", "I absolutely detest this course."])

::: {#exm-wine-reviews-3 style="background-color: #D5D1D164; padding: 20px"}

### Wine reviews sentiments
\index{Wine Reviews!Sentiment Analysis}

The number of reviews we have is close to 120,000. Hence, computing the
sentiments for each and every one will take a long time. Instead, we shall
compute the sentiments for a sample (of size 20, where possible) from each
variety of wine.

The following snippet samples 20 reviews from each wine type.

In [ ]:
tmp_df = wine_reviews.head(0).copy()

for x,vv in wine_reviews.groupby(wine_reviews.variety):
    grp_len = vv.shape[0]
    if(grp_len >= 20):
        vv = vv.sample(n=20, random_state=99)
    tmp_df = pd.concat([tmp_df, vv], ignore_index=True)
    
review_list = list(tmp_df.description)

The next snippet computes the sentiment scores for those sampled reviews.

In [ ]:
tmp_df['score'] = 0.00
tmp_df['label'] = ''
for i,rr in enumerate(review_list):
    tmp = classifier(rr)[0]
    tmp_df.loc[i, 'score'] = tmp['score']
    tmp_df.loc[i, 'label'] = tmp['label']

In [ ]:
from tqdm import tqdm

tmp_df['score'] = 0.00
tmp_df['label'] = ''

for i, rr in tqdm(enumerate(review_list), total=len(review_list), desc="Classifying reviews"):
    tmp = classifier(rr)[0]
    tmp_df.loc[i, 'score'] = tmp['score']
    tmp_df.loc[i, 'label'] = tmp['label']

This next snippet tabulates the sentiment classifications for the wine types.

In [ ]:
sent_counts = pd.crosstab(tmp_df.variety, tmp_df.label, margins=True)
sent_counts['proportion'] = sent_counts.POSITIVE/sent_counts.All

::: {.content-visible when-format="html"}

In [ ]:
show(sent_counts)

:::

::: {.content-visible when-format="pdf"}

In [ ]:
sent_counts.head()

:::

These are the reviews for one of the varieties that had a proportion of positive
reviews close to 50%.

In [ ]:
for x in wine_reviews[wine_reviews.variety == 'Tempranillo Blanco'].description.values:
    pp.pprint(x) 

The corresponding labels from the classifer are :

In [ ]:
tmp_df[tmp_df.variety == 'Tempranillo Blanco'][['description', 'label']]

:::

::: {.callout-note}
Do you agree with the classifications above? What would you investigate next?
:::

### Information Retrieval

In the NLP context, Information Retrieval (IR) refers to the task of returning
the most relevant set of documents, when given a query string. Search engines,
e.g. Google, are trained to perform fast and accurate IR. Typically, a long
list of documents is returned, with the most relevant one on top.

::: {.callout-note}
Pause for a moment, and consider how you would assess the performance of such a
search engine.
:::

::: {#exm-wine-reviews-2 style="background-color: #D5D1D164; padding: 20px"}

### Wine reviews information retrieval
\index{Wine Reviews!Information retrieval}

A simple way to perform IR is to use cosine similarity to compute how close the
given query vector is to the individual documents in the corpus.

The next snippet initialises a model for retrieving similar documents.

In [7]:
# IR retrieval use-case:
wine_tfidf = TfidfVectorizer(stop_words='english')
X2 = wine_tfidf.fit_transform(wine_reviews.description)

wine_nbrs = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='auto').fit(X2)

awc = wine_tfidf.transform(['acidic white chardonnay'])

distances, indices = wine_nbrs.kneighbors(awc, n_neighbors=8)

indices[0]

for x in indices[0]:
    pp.pprint(all_review_strings[x]) 

('Dry and acidic, this Chardonnay has a herbaceous earthiness, plus flavors of '
 'orange and pear.')
('A standard Chardonnay, dry and nicely acidic, with citrus, pear, vanilla, '
 'lees and oak flavors.')
('Pungent up front, with green herb, white pepper and citrus aromas, this is '
 'zesty and acidic on the palate, with a monotonous lemon flavor on the '
 'finish. It turns more tart and acidic as it airs.')
'This is thin and acidic, with flavors of sour cherry candy and spice.'
'This is acidic and sweet, with a medicinal taste.'
('Dry, acidic and earthy, lacking the richness you want in a fine Chardonnay. '
 'Could almost be a Pinot Grigio, with its crisp citrus and mineral flavors.')
('On the nose, the red berry aromas are rough and scratchy. The palate feels '
 'acidic and clipped, with tart flavors of red plum, cranberry and pie cherry. '
 'The finish is long and acidic.')
('This Chardonnay has vanilla, lemon blossom and peach aromas. Lemon curd and '
 'green apple flavors come th

In [ ]:
dct = gensim.corpora.Dictionary(all_strings_tokenized)
bow_corpus = [dct.doc2bow(text) for text in all_strings_tokenized]
tfidf = gensim.models.TfidfModel(dictionary=dct)

NLP corpora are typically very large. Before we can find matching documents, we
build a similarity index, so that matches are returned quicker. We try
something simple at first:

> Which documents/reviews are similar to the first one?

In [ ]:
index = gensim.similarities.Similarity(None, 
  corpus=tfidf[bow_corpus], num_features=len(dct))
sims = index[tfidf[bow_corpus[0]]]

These are the most similar reviews to _review id 0_. Of course, the first review
itself is there! Let's retrieve and print all the reviews similar to the first
one.

In [ ]:
for x in np.argsort(-sims)[:5]:
    pp.pprint(all_review_strings[x]) 

Now we try a new query of our own: "acidic chardonnay". First we preprocess it,
like we did the original documents.

In [ ]:
q1 = [wn.lemmatize(x) for x in preprocess_string('acidic white chardonnay', CUSTOM_FILTER)]
sims = index[tfidf[dct.doc2bow(q1)]]

Now we print the top 5 most similar reviews to our query.

In [ ]:
q1_results = np.argsort(-sims)[:5]

for x in q1_results:
    pp.pprint(all_review_strings[x]) 

:::

::: {.callout-note}
Try your favourite tastes, see if you discover a wine you like/dislike 🍷🍇🥂
:::

### Topic Modeling

The LDA (Latent Dirichlet Allocation) model assumes the following intuitive
generative process for the documents:

1. There is a set of $K$ topics that the documents come from. Each document
   contains words from several topics. There is a probability mass function on the
   topics for each document.
2. For each topic, there is a probability mass function for the distribution of
   words in that topic. 

At the end of LDA topic modeling, we will be able to tell, for a particular (new
or old) document: the weight combination of the topics for that document. For
each topic, we would be able to tell the terms that are salient. LDA only gives
us the probabilistic weights - we have to interpret them ourselves.

::: {#exm-wine-reviews-lda style="background-color: #D5D1D164; padding: 20px"}

### Wine reviews topic modeling
\index{Wine Reviews!Topic modeling}

Suppose we decide to split the corpus into 10 topics. Let us investigate what
these topics consist of.

Take note that batch learning uses less memory than online learning.

In [8]:
# Step 3: Train LDA model
lda_obj = LatentDirichletAllocation(
    n_components=10,           # Number of topics
    random_state=41,
    max_iter=50,
    learning_method='batch',
    evaluate_every=2,
    verbose=1,
    n_jobs=2               # Use all CPU cores
)

In [9]:
lda_obj.fit(X2)

iteration: 1 of max_iter: 50
iteration: 2 of max_iter: 50, perplexity: 7252.7861
iteration: 3 of max_iter: 50
iteration: 4 of max_iter: 50, perplexity: 6174.4718
iteration: 5 of max_iter: 50
iteration: 6 of max_iter: 50, perplexity: 5627.1583
iteration: 7 of max_iter: 50
iteration: 8 of max_iter: 50, perplexity: 5377.4954
iteration: 9 of max_iter: 50
iteration: 10 of max_iter: 50, perplexity: 5311.5485
iteration: 11 of max_iter: 50
iteration: 12 of max_iter: 50, perplexity: 5287.9484
iteration: 13 of max_iter: 50
iteration: 14 of max_iter: 50, perplexity: 5277.8096
iteration: 15 of max_iter: 50
iteration: 16 of max_iter: 50, perplexity: 5272.0119
iteration: 17 of max_iter: 50
iteration: 18 of max_iter: 50, perplexity: 5268.6634
iteration: 19 of max_iter: 50
iteration: 20 of max_iter: 50, perplexity: 5266.1550
iteration: 21 of max_iter: 50
iteration: 22 of max_iter: 50, perplexity: 5264.2939
iteration: 23 of max_iter: 50
iteration: 24 of max_iter: 50, perplexity: 5263.1171
iteration: 25

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",50
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",2
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [ ]:
lda_obj.

In [ ]:
# %pip install --no-deps pyLDAvis funcy

In [10]:
import pyLDAvis.lda_model as lda_model

In [11]:
vis = lda_model.prepare(
    lda_obj,
    X2,
    wine_tfidf
)

In [13]:
# Display visualization
import pyLDAvis
pyLDAvis.display(vis)

In [ ]:
lda1 = gensim.models.LdaModel(corpus= bow_corpus, num_topics=10, 
                              id2word=dct, random_state=41)
reviews_vis_data = gensimvis.prepare(lda1, bow_corpus, dct)

pp.pprint(lda1.show_topics())

The output provides the most common terms that define each topic (remember:
each topic is defined as a *probability distribution over the vocabulary*). 

We can also find out which *topics* a particular document is distributed over.
For instance, the output below shows that words in document 0 are predominantly
drawn from topic 3.

In [ ]:
lda1.get_document_topics(bow_corpus[0])

In [ ]:
pp.pprint(all_review_strings[0])

:::

A delightful visualisation from `pyLDAvis` allows us to
understand the "distance" between topics, and the frequent words from each topic
easily. To generate the visualisation in your notebook, execute the following 
command.

In [ ]:
#| eval: false
pyLDAvis.display(reviews_vis_data)

The online version of our textbook contains the interactive version.

::: {.content-visible when-format="html"}

* [Interactive visualisation](04-nlp_ldavis.html)

:::

::: {.content-visible when-format="pdf"}
Here is a static version for the pdf textbook:

![LDA visualisation](figs/ldavis_random_state_41.png){align="center"}
:::

A few things to note:

1. The numbering of topics in the text version of the topics does not match the 
   numbers within the circles of the visualisation. For easy mapping, we have the
   following mapping of text topics to visual circles:
   * 0 -> 7, 1 -> 1, 2 -> 6, 3 -> 4, 4 -> 3, 
     5 -> 8, 6 -> 9, 7 -> 2, 8 -> 5, 9 -> 10
2. The circles in the visualisation represent the topics. The distance between
   circles is the distance between the topic distributions, projected onto $R^2$ 
   using MDS. From the visual, we can see that topics 6 and 3 are very similar, 
   and so too are topics 4 and 5, and topics 1 and 8. It may be worth re-running
   the algorithm with a smaller number of topics.
3. If you inspect the text version of topics, you will notice many words repeated
   across topics, e.g. fruits, flavors, wine and palate. These should be removed
   before re-running the algorithm. Doing so would result in more distinct topics.
4. In terms of interpretation, checking against the topic distribution, it
   appears topics 4, 5, and 9 from the visual relate to acidic white wines with a
   citrus/fruit taste. Topics 1, 8, 2 and 7 correspond to strong tasting red wines.
   Topics 3 and 6 describe aromatic wines.
   taste
   
## Interpretation of Neural Models

Neural models have achieved impressive performance on a number of
language-related tasks. However, one criticism of them is that they are
"black-box" models; we do not fully grasp how they work. This can lead to a
mistrust of such models, with good reason. If we do not fully know how these
models work, we would not know when they are might fail, or we might not know
the reason when they do fail (or make an incorrect prediction). For this
reason, a huge amount of research effort is currently directed towards
understanding and interpreting neural models.

One approach is to identify which examples in the training set were most
influential for predictions regarding particular test instances. For incorrect
predictions, this could give us intuition on why the model is failing, and
guide us to ways to fix it. Here is an example where it was possible to
pinpoint why a model yielded incorrect sentiment prediction.

![@han2020explaining](figs/explaining_black_box_predictions.png){#fig-black-box width=70%}

In the example shown in @fig-black-box, the statement "A sometimes tedious film"
has been wrongly classified as having a "positive" sentiment. To understand why
this happened, training instances are removed one at a time; those that result
in the greatest change in the prediction are deemed to be influential points.
These are shown on the right side of the diagram. The presence of multiple 
ambivalent statements that were labelled as "positive" makes it easier to 
understand why the mistake happened.

Another approach is to identify which parts of the test sentence itself were
important to the eventual prediction. Imagine perturbing the test sentence in
some ways, and studying how the prediction changed. In one study of a
Question-Answering model (see @fig-infl-test-cases), the question was modified
by dropping the least important word, until the question was answered
incorrectly. This gives us an indication of which words were vital for the
question to be answered correctly. In this case, the study revealed something
pathological about the model. A one-word question seemed to still provide the 
correct answer from this passage!

![@sun2021interpreting](figs/pathologies_03.png){width=80% #fig-infl-test-cases}

For transformers in particular, a great deal of study has focused on the
weights that the attention layers pick up. By relating these to linguistic
information, researchers try to infer the precise language-related information
that models retain. For instance, it has been found that BERT learns parts of
speech. It is also able to identify the dependencies between words (see @fig-bert-dep).

![@clark2019does](figs/what_does_bert_clark_preposition.png){width=80% #fig-bert-dep}

## References

### Video explainers

1. [Transformer models and BERT](https://www.youtube.com/watch?v=t45S_MwAcOw): A very good video from Google Cloud Tech on current neural models (11:37)
2. [Introduction to RNN](https://www.youtube.com/watch?v=UNmqTiOnRfg)
3. [Introduction to BERT](https://www.youtube.com/watch?v=ioGry-89gqE)


### Website References

1. [Hugging Face course on transformers](https://huggingface.co/learn/nlp-course/chapter1/1)
2. [Gensim documentation](https://radimrehurek.com/gensim/auto_examples/index.html): Contains tutorials as well.
3. [Using sklearn to perform LDA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html): We can also use scikit-learn
   to perform LDA.
4. [Visualising LDA](https://github.com/bmabey/pyLDAvis): Contains sample notebooks for the visualisation.